# Kernels masterclass - Colab bootstrap

`Runtime -> Change runtime type -> T4 GPU` first, then run the cells top to bottom.

Cell 1 clones and reports the machine. Cell 2 gives you `work('02-vector-add')` to jump into any exercise, and `run()` / `sanitize()` / `profile()` to drive it.

In [ ]:
!nvidia-smi
!nvcc --version

import os
REPO = '/content/Machine-Learning'
if not os.path.isdir(REPO):
    !git clone --depth 1 https://github.com/DorsaRoh/Machine-Learning.git {REPO}
else:
    !cd {REPO} && git pull --ff-only
KERNELS = os.path.join(REPO, 'Kernels')
print('course at', KERNELS)

In [ ]:
import os, subprocess, textwrap

def _sh(cmd, cwd=None):
    p = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    print(p.stdout + p.stderr)
    return p.returncode

CURRENT = {'dir': None}

def exercises():
    root = os.path.join(KERNELS, 'exercises')
    for name in sorted(os.listdir(root)):
        if os.path.isdir(os.path.join(root, name)):
            print(name)

def work(name):
    """Select an exercise, print its brief, and open the kernel skeleton."""
    d = os.path.join(KERNELS, 'exercises', name)
    assert os.path.isdir(d), f'no such exercise: {name}'
    CURRENT['dir'] = d
    brief = os.path.join(d, 'README.md')
    if os.path.exists(brief):
        print(open(brief).read())
    for skel in ('kernel.cu', 'kernels.py', 'ext.py'):
        p = os.path.join(d, skel)
        if os.path.exists(p):
            print(f'\n--- {skel} ---\n')
            print(open(p).read())
    return d

def write_kernel(src):
    """Overwrite kernel.cu for the selected exercise from a notebook cell."""
    with open(os.path.join(CURRENT['dir'], 'kernel.cu'), 'w') as f:
        f.write(textwrap.dedent(src))

def run(args=''):
    d = CURRENT['dir']
    # Parts 0-V are make-driven; the Part VI exercises are Python.
    if os.path.exists(os.path.join(d, 'Makefile')):
        return _sh(f'make run ARGS="{args}"', d)
    return _sh(f'python bench.py {args}', d)
def sanitize(args=''): return _sh(f'make sanitize ARGS="{args}"', CURRENT['dir'])
def profile(args=''):  return _sh(f'make profile ARGS="{args}"', CURRENT['dir'])
def sass():            return _sh('cuobjdump -sass exercise', CURRENT['dir'])

exercises()

In [ ]:
work('00-hello-gpu')
run()

## Editing kernels

Two workflows, pick one:

1. **Notebook-native**: call `write_kernel('''...''')` with your full `kernel.cu`, then `run()`.
2. **Real editor** (better): work in this repo locally in Cursor, `git push`, then re-run cell 1 to pull. You keep your history, and your kernels survive the runtime being recycled.

Colab runtimes die. Commit your kernels.